# Tutorial 10: Sparse Autoencoders

**Prerequisites:** T03 (Residual Stream), T05 (MLPs and Features)

**What you'll learn:**
- Why sparse autoencoders (SAEs) are important for interpretability
- How SAEs work: architecture, training, and feature discovery
- How to train an SAE on model activations
- How to interpret the features an SAE discovers

---

## The Superposition Problem

In Tutorial T05, we saw that individual neurons are often **polysemantic** — they respond to multiple unrelated concepts. This happens because models use **superposition**: they represent more features than they have dimensions by encoding them in overlapping, nearly-orthogonal directions.

A residual stream of dimension 768 might represent thousands of features ("is a number", "is a proper noun", "is in a list", etc.) using directions that are almost but not exactly orthogonal.

**Sparse autoencoders** aim to untangle this: they learn a larger set of directions ("features") that are more interpretable than individual neurons.

## How SAEs Work

```
Input:    x (model activation, dimension d_model)
Encode:   features = ReLU(x @ W_enc + b_enc)   # d_model → n_features (sparse!)
Decode:   x_hat = features @ W_dec + b_dec      # n_features → d_model
Loss:     ||x - x_hat||² + λ * ||features||₁
```

The key is the **L1 penalty** on features: it forces most features to be zero (sparse), so only a few features fire for any given input. Each feature direction in `W_dec` should correspond to an interpretable concept.

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.sae import SparseAutoencoder, train_sae

cfg = HookedTransformerConfig(
    n_layers=2, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
print(f'Model ready: d_model={cfg.d_model}')

## Step 1: Collect Training Data

SAEs are trained on activations from a specific hook point. Let's collect activations from the residual stream after layer 0.

In [ ]:
# Generate training data: activations from many random inputs
rng = jax.random.PRNGKey(0)
hook_name = 'blocks.0.hook_resid_post'

all_activations = []
n_examples = 200

for i in range(n_examples):
    rng, subkey = jax.random.split(rng)
    random_tokens = jax.random.randint(subkey, (8,), 0, cfg.d_vocab)
    _, cache = model.run_with_cache(random_tokens)
    acts = cache[hook_name]  # [seq, d_model]
    all_activations.append(np.array(acts))

# Stack into a single matrix
activations = np.concatenate(all_activations, axis=0)  # [n_total, d_model]
print(f'Collected {activations.shape[0]} activation vectors of dimension {activations.shape[1]}')

## Step 2: Create and Train the SAE

In [ ]:
# Create an SAE with 4x overcomplete dictionary
n_features = cfg.d_model * 4  # 128 features for 32-dim activations

sae = SparseAutoencoder(
    d_model=cfg.d_model,
    n_features=n_features,
    key=jax.random.PRNGKey(42),
)
print(f'SAE architecture:')
print(f'  Input dimension:   {cfg.d_model}')
print(f'  Feature dimension: {n_features} ({n_features // cfg.d_model}x overcomplete)')
print(f'  W_enc shape: {sae.W_enc.shape}  [d_model, n_features]')
print(f'  W_dec shape: {sae.W_dec.shape}  [n_features, d_model]')

In [ ]:
# Train the SAE
activations_jax = jnp.array(activations)

trained_sae, losses = train_sae(
    sae, activations_jax,
    n_epochs=50,
    lr=1e-3,
    l1_coeff=0.01,  # Sparsity penalty
    batch_size=64,
)

print(f'Training complete!')
print(f'  Initial loss: {losses[0]:.4f}')
print(f'  Final loss:   {losses[-1]:.4f}')

## Step 3: Analyze the Learned Features

In [ ]:
# Encode some activations and look at the feature activations
test_tokens = jnp.array([1, 42, 17, 88, 55])
_, test_cache = model.run_with_cache(test_tokens)
test_acts = test_cache[hook_name]  # [seq, d_model]

# Encode through the SAE
feature_acts = trained_sae.encode(test_acts)  # [seq, n_features]
reconstruction = trained_sae.decode(feature_acts)  # [seq, d_model]

# How good is the reconstruction?
recon_error = float(jnp.mean((test_acts - reconstruction) ** 2))
original_var = float(jnp.var(test_acts))
r_squared = 1 - recon_error / original_var

print(f'Reconstruction quality:')
print(f'  MSE:       {recon_error:.6f}')
print(f'  R²:        {r_squared:.4f}')
print(f'\nFeature sparsity:')

for pos in range(len(test_tokens)):
    fa = np.array(feature_acts[pos])
    n_active = int(np.sum(fa > 0.01))
    top_feature = int(np.argmax(fa))
    top_val = float(fa[top_feature])
    print(f'  Position {pos} (token {int(test_tokens[pos]):2d}): '
          f'{n_active}/{n_features} features active, '
          f'strongest = feature {top_feature} ({top_val:.3f})')

In [ ]:
# Analyze individual features: what does each feature "mean"?
# The decoder weights tell us what direction each feature corresponds to.

W_U = np.array(model.unembed.W_U)

# For the top features that fired, what tokens do they promote?
top_features = np.argsort(np.array(feature_acts[-1]))[::-1][:5]

print(f'Top 5 active features at last position:')
for feat in top_features:
    feat_dir = np.array(trained_sae.W_dec[feat])  # [d_model]
    logit_effects = feat_dir @ W_U  # [d_vocab]
    
    promoted = np.argsort(logit_effects)[::-1][:3]
    suppressed = np.argsort(logit_effects)[:3]
    
    act_val = float(feature_acts[-1, feat])
    promoted_str = ', '.join([f'tok {t}({logit_effects[t]:+.2f})' for t in promoted])
    
    print(f'\n  Feature {feat} (activation = {act_val:.3f}):')
    print(f'    Promotes: {promoted_str}')
    suppressed_str = ', '.join([f'tok {t}({logit_effects[t]:+.2f})' for t in suppressed])
    print(f'    Suppresses: {suppressed_str}')

In [ ]:
# Feature activation patterns: which tokens activate each feature?
from irtk.sae import top_activating_examples

# Generate a dataset of activations with their token contexts
rng = jax.random.PRNGKey(99)
example_tokens = []
example_acts = []

for i in range(100):
    rng, subkey = jax.random.split(rng)
    t = jax.random.randint(subkey, (5,), 0, cfg.d_vocab)
    _, c = model.run_with_cache(t)
    example_tokens.append(np.array(t))
    example_acts.append(np.array(c[hook_name]))

# Find top activating examples for each of our top features
results = top_activating_examples(
    trained_sae, example_acts, example_tokens, top_k=3,
)

print('Top activating examples for selected features:')
for feat in top_features[:3]:
    print(f'\n  Feature {feat}:')
    if feat < len(results):
        for ex in results[feat][:3]:
            print(f'    tokens={list(ex["tokens"])}, '
                  f'pos={ex["position"]}, '
                  f'activation={ex["activation"]:.3f}')

## Using SAEs for Intervention

SAE features can be used for fine-grained interventions — clamping individual features on or off.

In [ ]:
# Clamp a specific feature to zero and see the effect
target_feature = top_features[0]

def clamp_feature(x, name, sae=trained_sae, feat=target_feature):
    """Zero out a specific SAE feature in the activation."""
    # Encode
    features = sae.encode(x)
    # Zero the target feature
    features = features.at[:, feat].set(0.0)
    # Decode back
    return sae.decode(features)

logits_base = model(test_tokens)
logits_clamped = model.run_with_hooks(
    test_tokens, fwd_hooks={hook_name: clamp_feature}
)

diff = float(jnp.linalg.norm(logits_clamped[-1] - logits_base[-1]))
base_pred = int(jnp.argmax(logits_base[-1]))
new_pred = int(jnp.argmax(logits_clamped[-1]))

print(f'Effect of clamping feature {target_feature} to zero:')
print(f'  Logit difference: {diff:.4f}')
print(f'  Base prediction: token {base_pred}')
print(f'  New prediction:  token {new_pred}')
print(f'  Prediction changed: {base_pred != new_pred}')

## Key Takeaways

1. **Superposition** means models represent more concepts than they have dimensions
2. **SAEs** learn an overcomplete dictionary of sparse, interpretable features
3. **Training**: encode with ReLU + L1 sparsity → decode → minimize reconstruction error
4. **Feature interpretation**: look at decoder weights to see what each feature promotes in the output
5. **Top activating examples** show what inputs trigger each feature
6. **Feature-level intervention** enables finer-grained causal analysis than component-level ablation

**Next: [T11 — Advanced Techniques](T11_advanced_techniques.ipynb)** — Steering, probing, and beyond.